In [32]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [33]:
def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    for files in pdf_files:
        print(files.name)
        try:
            loader = PyMuPDFLoader(str(files))
            document = loader.load()
            for doc in document:
                doc.metadata["Source_File"] = files.name
                doc.metadata["file_type"] = "pdf"
            all_documents.extend(document)
            print(len(document))

        except Exception as e:
            print(f"error, {e}")

    print(len(all_documents))
    return all_documents

docs = process_all_pdfs("../data/pdf_files")

research_papers_merged.pdf
38
38


In [34]:
def split_documents(docs, chunk_size = 1000, chunk_overlap = 200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n"," ", "  "]
    )

    split_docs = text_splitter.split_documents(docs)
    print(len(split_docs))

    return split_docs

In [35]:
split_docs = split_documents(docs)

182


In [44]:
split_docs

[Document(metadata={'producer': 'iLovePDF', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf_files\\research_papers_merged.pdf', 'file_path': '..\\data\\pdf_files\\research_papers_merged.pdf', 'total_pages': 38, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-09-15T14:11:03+00:00', 'trapped': '', 'modDate': 'D:20260915141103Z', 'creationDate': '', 'page': 0, 'Source_File': 'research_papers_merged.pdf', 'file_type': 'pdf'}, page_content='Biomimetic Intelligence and Robotics 4 (2024) 100148\nContents lists available at ScienceDirect\nBiomimetic Intelligence and Robotics\njournal homepage: www.elsevier.com/locate/birob\nResearch Article\nLocomotion control of a rigid-soft coupled snake robot in multiple\nenvironments\nXuanyi Zhou a,b, Yuqiu Zhang c, Zhiwei Qiu a, Zhecheng Shan a, Shibo Cai a, Guanjun Bao a,∗\na College of Mechanical Engineering, Zhejiang University of Technology, Hangzhou 310023, China\nb State Key Laboratory of C

In [36]:
import numpy as np
from sentence_transformers import SentenceTransformer 
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [37]:
class EmbeddingManager:
    def __init__(self, model = "all-MiniLM-L6-v2"):
        self.model_name = model 
        self.model = None
        self.load_model()

    def load_model(self):
        try:
            self.model = SentenceTransformer(self.model_name)
        except Exception as e:
            print("Error", e)

    def get_embedding_dimensions(self):
        if not self.model:
            raise ValueError("model not loaded")
        return self.model.get_sentence_embedding_dimension() 

    def generate_embeddings(self, texts):
        if not self.model:
            raise ValueError("model not loaded")

        embeddings = self.model.encode(texts, show_progress_bar = True)
        print(len(embeddings))
        return embeddings


In [45]:
embedding_manager = EmbeddingManager()
embeddings = embedding_manager.generate_embeddings([doc.page_content for doc in split_docs])

Batches: 100%|██████████| 6/6 [00:05<00:00,  1.02it/s]

182


In [48]:
import uuid

In [50]:
class VectorStore:
    def __init__(self, collection_name = "pdf_documents", persist_directory = "../vectorstore_db"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self.initialize_store()

    def initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path = self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description" : "PDF document embeddings for RAG"}
            )

            print(self.collection_name)
            print(self.collection.count())

        except Exception as e:
            print("Error", e)

    def add_document(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match the number of embeddings")

        ids = []
        metadatas = []
        document_text = []
        document_embeddings = []

        for i, (doc, embeddings) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            document_text.append(doc.page_content)
            document_embeddings.append(embeddings.tolist())

        try:
            self.collection.add(
                ids = ids,
                metadatas = metadatas,
                documents = document_text,
                embeddings = document_embeddings
            )
            print(self.collection.count())
        except Exception as e:
            print("Error", e)

vectorstore = VectorStore()
vectorstore.add_document(split_docs, embeddings)

pdf_documents
0
182


Retrival Pipeline

In [61]:
class RagRetriver:
    def __init__(self, vector_store, embedding_manager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrive(self, query, top_k = 5, score_threshold = 0):
        query_embeddings = self.embedding_manager.generate_embeddings(query)
        try:
            result = self.vector_store.collection.query(
                query_embeddings = [query_embeddings.tolist()],
                n_results = top_k
            )

            retrived_docs = []

            if result["documents"] and result["documents"][0]:
                documents = result["documents"][0]
                metadatas = result["metadatas"][0]
                distances = result["distances"][0]
                ids = result["ids"][0]

                for i, (id, docs, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrived_docs.append({
                            "id" : id,
                            "content" : docs,
                            "metadata" : metadata,
                            "similarity_score" : similarity_score,
                            "distance" : distance,
                            "rank" : i + 1
                        })

            print(len(retrived_docs))
            return retrived_docs
        except Exception as e:
            print("Error", e)
            return []

In [63]:
rag_retriver = RagRetriver(vectorstore, embedding_manager)
results = rag_retriver.retrive("what is snake robots")

Batches: 100%|██████████| 1/1 [00:00<00:00, 40.38it/s]

384
5


In [73]:
print(results[3]["content"])

536–547.
[12] A. Maity, S. Majumder, Implementation of serpentine locomotion, Int. J.
Intell. Syst. Technol. Appl. 11 (1–2) (2012) 81–101.
[13] P. Liljebäck, K.Y. Pettersen, Ø. Stavdahl, J.T. Gravdahl, Snake Robots:
Modelling, Mechatronics, and Control, Springer, 2013.
[14] J. Liu, Y. Tong, J. Liu, Review of snake robots in constrained environments,
Robot. Auton. Syst. 141 (2021) 103785.
[15] J.K. Hopkins, B.W. Spranklin, S.K. Gupta, A survey of snake-inspired robot
designs, Bioinspiration Biomim. 4 (2) (2009) 021001.
[16] A.A. Transeth, P. Liljeback, K.Y. Pettersen, Snake robot obstacle aided
locomotion: An experimental validation of a non-smooth modeling ap-
proach, in: 2007 IEEE/RSJ International Conference on Intelligent Robots
and Systems, 2007, pp. 2582–2589.
[17] C. Wright, A. Johnson, A. Peck, Z. Mccord, A. Naaktgeboren, P. Gianfortoni,
M. Gonzalez-Rivero, R. Hatton, H. Choset, Design of a modular snake
